# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aisyahnabillah/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: start with a Decision Tree (readable, max_depth=3), then compare against a Random 
Forest to see whether the extra complexity earns its keep.

Why this fits: my question shape is "which pages should be reviewed first?", a ranking 
question. Per the toolkit, that means using a classifier's probability output, evaluated at 
precision@K, not raw accuracy. This is the same approach I already used in ML-01 (decision 
tree vs hand rule), now compared against my own baseline from ML-07 instead of a generic 
starter rule.

Target/proxy: is_declining_label = trend_direction == "down", the same observed-outcome label 
used throughout this track. trend_direction and trend_pct are excluded from the feature set, 
since the label is derived from them (the leakage lesson from ML-04).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Recreate my Week-4 baseline score, same formula as w04_baseline_score.ipynb
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["baseline_action_score"] = df["stale"] * df["visible"] * df["impressions_90d"]

# Target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Honest features only — trend_direction / trend_pct excluded (label leakage, per ML-04)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume", "engagement_rate"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

print(df[["stale", "visible", "baseline_action_score", "is_declining_label"]].describe())

              stale       visible  baseline_action_score  is_declining_label
count  30000.000000  30000.000000           30000.000000        30000.000000
mean       0.005800      0.557533               6.573700            0.542067
std        0.075938      0.496687             528.117366            0.498236
min        0.000000      0.000000               0.000000            0.000000
25%        0.000000      0.000000               0.000000            0.000000
50%        0.000000      1.000000               0.000000            1.000000
75%        0.000000      1.000000               0.000000            1.000000
max        1.000000      1.000000           61678.000000            1.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-grouped holdout (GroupShuffleSplit on client_id), not a plain random split. 
This matters because pages from the same client can share patterns (writing style, industry, 
existing SEO health) that a random split would let leak between train and test, making the 
model look better than it really is. This is the same lesson from ML-01/02, where switching 
from in-sample scoring to a client-holdout split dropped precision from 0.800 to 0.400, in-
sample numbers were optimistic. Baseline and model are both evaluated on the exact same held-
out test clients, so the comparison in Section 3 is fair.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df.iloc[test_idx]["baseline_action_score"]

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Test base rate (declining): {y_test.mean():.3f}")

Train rows: 19166, Test rows: 10834
Test base rate (declining): 0.559


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The baseline rule from Week 4 outperformed both models at precision@20 (0.65 vs 0.60 Decision Tree vs 0.45 Random Forest) and tied or beat them at precision@50 (0.64 vs 0.56 vs 0.56). All three beat the base rate of 0.559, meaning random selection would only get about 55.9% right by chance, so even the weaker methods here still add some value over guessing. But the honest finding is that a simple, human-readable rule beat two learned models on this data and split, directly echoing the lesson from ML-01: a sharp hand-written rule can be excellent at the very top of a ranked list, and more complexity does not guarantee a better result.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results = []
for name, scores in [("Baseline (Week 4 rule)", baseline_test.values),
                      ("Decision Tree", tree_scores),
                      ("Random Forest", rf_scores)]:
    row = {
        "method": name,
        "precision@20": precision_at_k(scores, y_test.values, 20),
        "precision@50": precision_at_k(scores, y_test.values, 50),
    }
    results.append(row)

comparison = pd.DataFrame(results)
comparison["base_rate"] = y_test.mean()
print(comparison)

                   method  precision@20  precision@50  base_rate
0  Baseline (Week 4 rule)          0.65          0.64   0.559442
1           Decision Tree          0.60          0.56   0.559442
2           Random Forest          0.45          0.56   0.559442


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

# 3 concrete wrong cases: model scored high but label says NOT declining (false positive)
X_test_labeled = X_test.copy()
X_test_labeled["rf_score"] = rf_scores
X_test_labeled["actual_label"] = y_test.values
wrong_cases = X_test_labeled[X_test_labeled["actual_label"] == 0].sort_values("rf_score", ascending=False).head(3)
print(wrong_cases)

impressions_90d           0.321333
avg_position              0.220911
content_age_days          0.191333
word_count                0.134948
days_since_last_update    0.061261
ctr                       0.042534
search_volume             0.019525
engagement_rate           0.008154
dtype: float64
       content_age_days  days_since_last_update  impressions_90d  \
22524               275                     104              870   
22526               280                     104             3445   
22461               280                     104              909   

       avg_position   ctr  word_count  search_volume  engagement_rate  \
22524          17.6  0.11      1492.0           10.0              0.0   
22526          39.0  0.09      1480.0           20.0             20.0   
22461          33.4  0.11      1415.0           20.0              0.0   

       rf_score  actual_label  
22524  0.781166             0  
22526  0.780265             0  
22461  0.779767             0  


Top features: impressions_90d (0.321), avg_position (0.221), and content_age_days (0.191) 
dominate the Random Forest's decisions. Sanity check: these make intuitive sense (traffic 
volume, ranking position, and content age are all plausible signals for review priority), and 
none is suspiciously close to 1.0, so there's no obvious sign of leakage. Notably, 
days_since_last_update, the exact signal my Week-4 baseline was built around, ranks only 5th 
in importance (0.061), consistent with the OPPOSITE verdict I found for that signal in ML-07's 
Signal 1 check.

3 wrong cases: all three false positives share content_age_days around 275-280 days and 
days_since_last_update of 104 days, a profile the model has learned to associate with decline. 
But their actual outcome was NOT declining. This suggests the model may be over-relying on the 
age/staleness combination as a decline signal, the same pattern ML-07 already showed was 
unreliable (OPPOSITE verdict, small sample). These pages may have other factors, like recent 
content quality or external backlinks, that this feature set doesn't capture, keeping them 
stable despite looking "at risk" on paper.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.